In [1]:
import autogen
import chromadb
from chromadb.utils import embedding_functions
import uuid
from datetime import datetime

class VectorMemory:
    def __init__(self, collection_name="agent_memories"):
        self.client = chromadb.Client()
        self.embedding_fn = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.create_collection(
            name=collection_name,
            embedding_function=self.embedding_fn
        )

    def store(self, content: str, context: str = ""):
        self.collection.add(
            documents=[content],
            metadatas=[{"context": context, "timestamp": str(datetime.now())}],
            ids=[str(uuid.uuid4())]
        )

    def retrieve(self, query: str, n_results: int = 3):
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        return results['documents'][0]

class AgentWithMemory:
    def __init__(self):
        self.memory = VectorMemory()
        self.config_list = [{"model": "gpt-4", "api_key": "your-key-here"}]
        
        self.assistant = autogen.AssistantAgent(
            name="assistant",
            llm_config={
                "config_list": self.config_list,
                "cache_seed": 42
            }
        )
        
        self.user_proxy = autogen.UserProxyAgent(
            name="user_proxy",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=1
        )

    def chat(self, message: str):
        # Retrieve relevant memories
        memories = self.memory.retrieve(message)
        context = f"Previous relevant information: {memories}\n\nCurrent query: {message}"
        
        # Chat with context
        self.user_proxy.initiate_chat(
            self.assistant,
            message=context
        )
        
        # Store new memory
        self.memory.store(
            content=self.assistant.last_message()["content"],
            context=message
        )
        
        return self.assistant.last_message()["content"]

d:\GitHub\AutoGen-notebooks\.venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [2]:
import autogen
import chromadb
import openai
from typing import List, Dict
import uuid
from datetime import datetime

class VectorMemory:
    def __init__(self, openai_api_key: str, collection_name: str = "agent_memories"):
        self.client = chromadb.Client()
        self.collection = self.client.create_collection(name=collection_name)
        openai.api_key = openai_api_key
        
    def _get_embedding(self, text: str) -> List[float]:
        response = openai.Embedding.create(
            input=text,
            model="text-embedding-ada-002"
        )
        return response['data'][0]['embedding']
    
    def store(self, content: str, context: str = ""):
        embedding = self._get_embedding(content)
        self.collection.add(
            embeddings=[embedding],
            documents=[content],
            metadatas=[{
                "context": context,
                "timestamp": str(datetime.now())
            }],
            ids=[str(uuid.uuid4())]
        )
    
    def retrieve(self, query: str, n_results: int = 3) -> List[str]:
        query_embedding = self._get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        return results['documents'][0]

class AutoGenWithMemory:
    def __init__(self, openai_api_key: str):
        self.memory = VectorMemory(openai_api_key)
        self.config_list = [{
            "model": "gpt-4",
            "api_key": openai_api_key
        }]
        
        self.assistant = autogen.AssistantAgent(
            name="assistant",
            llm_config={
                "config_list": self.config_list,
                "cache_seed": 42
            }
        )
        
        self.user_proxy = autogen.UserProxyAgent(
            name="user_proxy",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=1
        )
    
    def chat(self, message: str) -> str:
        # Retrieve relevant memories
        memories = self.memory.retrieve(message)
        context = f"Previous relevant context:\n{memories}\n\nCurrent query: {message}"
        
        # Chat with context
        self.user_proxy.initiate_chat(
            self.assistant,
            message=context
        )
        
        response = self.assistant.last_message()["content"]
        
        # Store new memory
        self.memory.store(content=response, context=message)
        
        return response

In [ ]:
import autogen
import chromadb
import requests
import numpy as np
from typing import List, Dict
import uuid
from datetime import datetime
import ollama

class OllamaEmbedding:
    # def __init__(self, base_url: str = "http://localhost:11434"):
    #     self.base_url = base_url
        
    def get_embedding(self, text: str) -> List[float]:
        response = ollama.embeddings(model="nomic-embed-text",prompt=text)
        return response['embedding']

class VectorMemory:
    def __init__(self, collection_name: str = "agent_memories"):
        self.client = chromadb.Client()
        self.collection = self.client.create_collection(name=collection_name)
        self.embedder = OllamaEmbedding()
    
    def store(self, content: str, context: str = ""):
        embedding = self.embedder.get_embedding(content)
        self.collection.add(
            embeddings=[embedding],
            documents=[content],
            metadatas=[{
                "context": context,
                "timestamp": str(datetime.now())
            }],
            ids=[str(uuid.uuid4())]
        )
    
    def retrieve(self, query: str, n_results: int = 3) -> List[str]:
        query_embedding = self.embedder.get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        return results['documents'][0]

class AutoGenWithMemory:
    def __init__(self, config_list: List[Dict]):
        self.memory = VectorMemory()
        
        self.assistant = autogen.AssistantAgent(
            name="assistant",
            system_message="You are helpfil assistant to help user with their queries.",
            llm_config={
                "config_list": config_list,
                "cache_seed": 42
            },
            code_execution_config=False

        )
        
        self.user_proxy = autogen.UserProxyAgent(
            name="user_proxy",
            human_input_mode="NEVER", 
            max_consecutive_auto_reply=1,
            code_execution_config=False,
            is_termination_msg= lambda msg: "TERMINATE" in msg["content"],
        )
    
    def chat(self, message: str) -> str:
        # Get relevant memories
        memories = self.memory.retrieve(message)
        
        # Build context
        context = (
            "Previous relevant information:\n"
            + '\n'.join(memories) + "\n\n"
            + f"Current query: {message}"
        )
        
        # Chat with context
        self.user_proxy.initiate_chat(
            self.assistant,
            message=context + " and Say the word TERMINATE."
        )
        
        response = self.assistant.last_message()["content"]
        
        # Store new memory
        self.memory.store(content=response, context=message)
        
        return response

d:\GitHub\AutoGen-notebooks\.venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [2]:
# from vector_memory import AutoGenWithMemory

# Configuration
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        'api_key': 'ollama',
    },
]

# Initialize agent
agent = AutoGenWithMemory(config_list)

# Example conversation
response1 = agent.chat("My name is anoop, I like pizaa")
print("Response 1:", response1)



user_proxy (to assistant):

Previous relevant information:


Current query: My name is anoop, I like pizaaSay the word TERMINATE.

--------------------------------------------------------------------------------
[autogen.oai.client: 12-25 22:22:22] {432} WARNING - Model llama3.2 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
assistant (to user_proxy):

Hello Anoop! It's nice to meet you. Since you mentioned pizza is one of your likes, I can definitely chat with you about it. What's your favorite type of pizza? Do you have a go-to topping or a favorite pizza place?

--------------------------------------------------------------------------------
user_proxy (to assistant):



--------------------------------------------------------------------------------
[autogen.oai.client: 12-25 22:22:44] {432} WARNING - Model llama3.2 is not found. The cost will be 0. In your config_list, add fi

In [3]:
# Follow-up with memory
response2 = agent.chat("What is my name?, What do I like?")
print("Response 2:", response2)

Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


user_proxy (to assistant):

Previous relevant information:
It seems like you've pressed the stop button! No worries, we can start again whenever you're ready.

To recap, I know that:

* Your name is Anoop.
* Pizza is one of your favorite foods.

Is there anything else you'd like to talk about regarding pizza or anything else entirely?

Current query: What is my name?, What do I like?Say the word TERMINATE.

--------------------------------------------------------------------------------
[autogen.oai.client: 12-25 22:24:23] {432} WARNING - Model llama3.2 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
assistant (to user_proxy):

TERMINATE.

Let's start fresh! Since we didn't have a chance to continue our conversation earlier, I'll need a bit more information from you. Unfortunately, I don't have any previous knowledge about your name or preferences beyond what you mentioned earlier 